In [14]:
import os
import pandas as pd
import mlflow
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.neighbors import KNeighborsClassifier
from dotenv import load_dotenv

load_dotenv()

estimator = KNeighborsClassifier(n_neighbors=3)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

initial_csv_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models_artifacts/data/initial_data.csv"
pipeline_initial_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models"

storage_options = {
    "key": os.getenv("AWS_ACCESS_KEY_ID"),
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs": {"endpoint_url": "https://storage.yandexcloud.net"},
}
df_new: pd.DataFrame = pd.read_csv(
    initial_csv_path, storage_options=storage_options
)
pipeline = mlflow.sklearn.load_model(pipeline_initial_path)
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

X = df_new.drop(columns=["target", "end_date"], errors="ignore")
y = df_new["target"]
X_trans = preprocessor.transform(X)

model = CatBoostClassifier(auto_class_weights=auto_class_weights)

array([0, 0, 1, ..., 0, 1, 0])

In [5]:
from util.test_load_data_from_db import load_data_from_db

df = load_data_from_db()

In [ ]:
from collections import defaultdict

from catboost import CatBoostClassifier
from optuna.integration import MLflowCallback
import numpy as np
from numpy import array, median
import optuna
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
import mlflow
import os
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

TRACKING_SERVER_URI = "http://127.0.0.1:5003"
mlflow.set_tracking_uri(TRACKING_SERVER_URI)
mlflow.set_registry_uri(TRACKING_SERVER_URI)

EXPERIMENT_NAME = "bayesian_search_expertiment"
RUN_NAME = "model_bayesian_search"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model_bayesian"

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = target  # ваш код здесь
stratify_column = target  # ваш код здесь
test_size = 0.2  # ваш код здесь

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    random_state=42,
    stratify=df[stratify_column],
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")


def objective(trial: optuna.Trial) -> float:
    param = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.001, 0.1, log=True
        ),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False,
    }

    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        # ваш код здесь #
        train_x = X_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        train_y = y_train.iloc[train_index]
        val_y = y_train.iloc[val_index]
        model.fit(
            train_x,
            train_y,
            eval_set=(val_x, val_y),
            early_stopping_rounds=50,
            verbose=False,
        )
        probas = model.predict_proba(val_x)[:, 1]
        prediction = model.predict(val_x)

        cm = confusion_matrix(val_y, prediction, normalize="all").ravel()
        err_1 = cm[1]
        err_2 = cm[2]
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)

        metrics["err1"].append(err_1)
        metrics["err2"].append(err_2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)

    # ваш код здесь #
    err_1 = median(array(metrics["err1"]))
    err_2 = median(array(metrics["err2"]))
    auc = median(array(metrics["auc"]))
    precision = median(array(metrics["precision"]))
    recall = median(array(metrics["recall"]))
    f1 = median(array(metrics["f1"]))
    logloss = median(array(metrics["logloss"]))

    # --- ИСПРАВЛЕНИЕ 1: Сохраняем все метрики в user_attr триала ---
    trial.set_user_attr("err1", err_1)
    trial.set_user_attr("err2", err_2)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("logloss", logloss)
    trial.set_user_attr("auc", auc)

    return auc


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id


with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflc = MLflowCallback(
        tracking_uri=TRACKING_SERVER_URI,
        metric_name="AUC",
        create_experiment=False,
        mlflow_kwargs={
            "experiment_id": experiment_id,
            "tags": {MLFLOW_PARENT_RUN_ID: run_id},
            "nested": True,
        },
    )
    mlflow.set_tag("mlflow.runName", RUN_NAME)

    study = optuna.create_study(
        sampler=optuna.samplers.TPESampler(),
        study_name=STUDY_NAME,
        storage=STUDY_DB_NAME,
        direction="maximize",
        load_if_exists=True,
    )
    study.optimize(objective, n_trials=10, callbacks=[mlflc])
    best_params = study.best_params
    best_auc = study.best_value

    # --- ИСПРАВЛЕНИЕ 2: Обучаем и логируем ИТОГОВУЮ модель ---
    # Добавляем базовые параметры, которые не участвовали в Optuna
    final_params = best_params.copy()
    final_params.update(
        {
            "loss_function": "Logloss",
            "task_type": "CPU",
            "random_seed": 0,
            "iterations": 300,
            "verbose": False,
        }
    )

    # Обучаем финальную модель на ВСЁМ X_train
    best_model = CatBoostClassifier(**final_params)
    best_model.fit(X_train, y_train)

    # Записываем метрики и параметры в родительский Run
    print(f"Best AUC: {best_auc}")
    print(f"Best params: {best_params}")
    mlflow.log_params(final_params)
    mlflow.log_metric("best_cv_auc", best_auc)

    # Сохраняем модель в MLflow и регистрируем в Model Registry
    mlflow.catboost.log_model(
        cb_model=best_model,
        artifact_path="cv",
        registered_model_name="catboost_model",
    )

    print(f"Number of finished trials: {len(study.trials)}")
    print(f"Best params: {best_params}")

Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)


/var/folders/9p/vkzz84_d4_l8l4s8mjml37740000gn/T/ipykernel_36625/3560251324.py:140: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-07-23 15:15:41,172] Using an existing study with name 'churn_model_bayesian' instead of creating a new one.
[I 2026-07-23 15:15:42,673] Trial 11 finished with value: 0.8265878209167434 and parameters: {'learning_rate': 0.015997672121836268, 'depth': 8, 'l2_leaf_reg': 2.0388248291754123, 'random_strength': 3.7088066488550795}. Best is trial 11 with value: 0.8265878209167434.


🏃 View run 11 at: http://127.0.0.1:5003/#/experiments/26/runs/8905a205dcbb4f5c8a39dbbd4c7c4966
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/26


[I 2026-07-23 15:15:44,938] Trial 12 finished with value: 0.8260021327129077 and parameters: {'learning_rate': 0.020005215103538435, 'depth': 8, 'l2_leaf_reg': 2.197785365667413, 'random_strength': 3.5702993036300503}. Best is trial 11 with value: 0.8265878209167434.


🏃 View run 12 at: http://127.0.0.1:5003/#/experiments/26/runs/e1ac20175b95441090e6bfa1725498c3
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/26


...

[I 2026-07-23 15:16:00,289] Trial 19 finished with value: 0.8272536831760847 and parameters: {'learning_rate': 0.0916831834034732, 'depth': 10, 'l2_leaf_reg': 3.538357354879745, 'random_strength': 4.404815845499331}. Best is trial 18 with value: 0.8278016092288305.


🏃 View run 19 at: http://127.0.0.1:5003/#/experiments/26/runs/54d0a2bfe67e4d99ab514c687a57e922
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/26


[I 2026-07-23 15:16:02,473] Trial 20 finished with value: 0.8280520414263325 and parameters: {'learning_rate': 0.09908571783539068, 'depth': 10, 'l2_leaf_reg': 3.7312773168357296, 'random_strength': 4.579627129630167}. Best is trial 20 with value: 0.8280520414263325.


🏃 View run 20 at: http://127.0.0.1:5003/#/experiments/26/runs/fd94b706690c44ac9931a129f798f1ed
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/26
Best AUC: 0.8280520414263325
Best params: {'learning_rate': 0.09908571783539068, 'depth': 10, 'l2_leaf_reg': 3.7312773168357296, 'random_strength': 4.579627129630167}


2026/07/23 15:16:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Взято из урока

In [ ]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("S3_ACCESS_KEY")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")


mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

EXPERIMENT_NAME = "model_bayesian_search"  # ваш код здесь
RUN_NAME = "model_bayesian_search"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model"


def objective(trial: optuna.Trial) -> float:
    param = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.001, 0.1, log=True
        ),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False,
    }
    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2)

    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        # ваш код здесь #
        train_x = X_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        train_y = y_train.iloc[train_index]
        val_y = y_train.iloc[val_index]
        model.fit(
            train_x,
            train_y,
            eval_set=(val_x, val_y),
            early_stopping_rounds=50,
            verbose=False,
        )
        probas = model.predict_proba(val_x)[:, 1]
        prediction = model.predict(val_x)
        _, err1, _, err2 = confusion_matrix(
            val_y, prediction, normalize="all"
        ).ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)

        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)

    # ваш код здесь #
    err_1 = median(array(metrics["err1"]))
    err_2 = median(array(metrics["err2"]))
    auc = median(array(metrics["auc"]))
    precision = median(array(metrics["precision"]))
    recall = median(array(metrics["recall"]))
    f1 = median(array(metrics["f1"]))
    logloss = median(array(metrics["logloss"]))

    return auc


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id


with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflc = MLflowCallback(
        tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}",
        metric_name="AUC",
        create_experiment=False,
        mlflow_kwargs={
            "experiment_id": experiment_id,
            "tags": {MLFLOW_PARENT_RUN_ID: run_id},
        },
    )
    mlflow.set_tag("mlflow.runName", RUN_NAME)

    study = optuna.create_study(
        sampler=optuna.samplers.TPESampler(),
        study_name=STUDY_NAME,
        storage=STUDY_DB_NAME,
        direction="maximize",
        load_if_exists=True,
    )
    study.optimize(objective, n_trials=10, callbacks=[mlflc])
    best_params = study.best_params
    best_auc = study.best_value

    # --- ИСПРАВЛЕНИЕ 2: Обучаем и логируем ИТОГОВУЮ модель ---
    # Добавляем базовые параметры, которые не участвовали в Optuna
    final_params = best_params.copy()
    final_params.update(
        {
            "loss_function": "Logloss",
            "task_type": "CPU",
            "random_seed": 0,
            "iterations": 300,
            "verbose": False,
        }
    )

    # Обучаем финальную модель на ВСЁМ X_train
    best_model = CatBoostClassifier(**final_params)
    best_model.fit(X_train, y_train)

    # Записываем метрики и параметры в родительский Run
    print(f"Best AUC: {best_auc}")
    print(f"Best params: {best_params}")
    mlflow.log_params(final_params)
    mlflow.log_metric("best_cv_auc", best_auc)

    # Сохраняем модель в MLflow и регистрируем в Model Registry
    mlflow.catboost.log_model(
        cb_model=best_model,
        artifact_path="model",
        registered_model_name="catboost_model",
    )

    print(f"Number of finished trials: {len(study.trials)}")
    print(f"Best params: {best_params}")